# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/data00077/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

Signal 1 — Staleness: CONFIRMED. Older content maps consistently to lower freshness tiers: rows at 90–179 days fall in the 91–180 freshness tier, while rows at 180+ days fall in the 181+ tier.

Signal 2 — Search volume: CONFIRMED. Higher 90-day impressions align with stronger impression tiers: 500–999 impressions are all moderate, while 1000+ impressions are moderate, good, or excellent.

Rule: Prioritize content that is at least 180 days since its last update and has at least 500 impressions in the last 90 days. The score is the 90-day impression count when both conditions are true, otherwise zero. The reason code is stale_but_visible and the action is REVIEW_REFRESH.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Signal 1: staleness vs FlyRank freshness tier
staleness_check = (
    df.assign(
        staleness_bucket=pd.cut(
            df["days_since_last_update"],
            bins=[-1, 89, 179, 364, float("inf")],
            labels=["0-89d", "90-179d", "180-364d", "365d+"]
        )
    )
    .groupby(
        ["staleness_bucket", "freshness_tier"],
        observed=False
    )
    .size()
    .reset_index(name="n")
)

print("SIGNAL 1 — STALENESS")
display(staleness_check)


# Signal 2: impressions vs FlyRank impression tier
volume_check = (
    df.assign(
        volume_bucket=pd.cut(
            df["impressions_90d"],
            bins=[-1, 0, 99, 499, 999, float("inf")],
            labels=["0", "1-99", "100-499", "500-999", "1000+"]
        )
    )
    .groupby(
        ["volume_bucket", "impression_tier"],
        observed=False
    )
    .size()
    .reset_index(name="n")
)

print("\nSIGNAL 2 — SEARCH VOLUME")
display(volume_check)

SIGNAL 1 — STALENESS


,staleness_bucket,freshness_tier,n
0,0-89d,0-30,20480
1,0-89d,181+,0
2,0-89d,31-90,175
3,0-89d,91-180,0
4,90-179d,0-30,0
5,90-179d,181+,0
6,90-179d,31-90,0
7,90-179d,91-180,9171
8,180-364d,0-30,0
9,180-364d,181+,169



SIGNAL 2 — SEARCH VOLUME


,volume_bucket,impression_tier,n
0,0,excellent,0
1,0,good,0
2,0,low,0
3,0,moderate,0
4,1-99,excellent,0
5,1-99,good,0
6,1-99,low,7994
7,1-99,moderate,0
8,100-499,excellent,0
9,100-499,good,0


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the transparent baseline score

baseline = df.copy()

# Two pre-decision signals
baseline["stale"] = (
    baseline["days_since_last_update"] >= 180
).astype(int)

baseline["visible"] = (
    baseline["impressions_90d"] >= 500
).astype(int)

# Score only when BOTH conditions are satisfied
baseline["score"] = (
    baseline["stale"]
    * baseline["visible"]
    * baseline["impressions_90d"]
)

# Assign reason and action consistently with the score
baseline["reason_code"] = "not_selected"

baseline.loc[
    baseline["score"] > 0,
    "reason_code"
] = "stale_but_visible"

baseline["action"] = "NO_ACTION"

baseline.loc[
    baseline["score"] > 0,
    "action"
] = "REVIEW_REFRESH"

# Rank highest-priority items first
baseline = baseline.sort_values(
    ["score", "impressions_90d"],
    ascending=[False, False]
).reset_index(drop=True)

# Output columns
queue_columns = [
    "content_id",
    "client_id",
    "days_since_last_update",
    "impressions_90d",
    "freshness_tier",
    "impression_tier",
    "score",
    "reason_code",
    "action"
]

baseline_queue = baseline[queue_columns].copy()

# Write the required CSV
output_path = "work/outputs/baseline_action_score.csv"

import os
os.makedirs("work/outputs", exist_ok=True)

baseline_queue.to_csv(output_path, index=False)

print("Ranked queue rows:", len(baseline_queue))
print("Positive-score rows:", (baseline_queue["score"] > 0).sum())
print("Zero-score rows:", (baseline_queue["score"] == 0).sum())

print("\nTop 20:")
display(baseline_queue.head(20))

print("\nCSV written to:", output_path)

Ranked queue rows: 30000
Positive-score rows: 17
Zero-score rows: 29983

Top 20:


,content_id,client_id,days_since_last_update,impressions_90d,freshness_tier,impression_tier,score,reason_code,action
0,content_cf56e2e2e282,client_7f2253d7e2,194,61678,181+,excellent,61678,stale_but_visible,REVIEW_REFRESH
1,content_7368877ea310,client_7f2253d7e2,194,59472,181+,excellent,59472,stale_but_visible,REVIEW_REFRESH
2,content_1bfaa38ff26c,client_7f2253d7e2,194,25715,181+,good,25715,stale_but_visible,REVIEW_REFRESH
3,content_0a91db491d14,client_7f2253d7e2,193,13299,181+,good,13299,stale_but_visible,REVIEW_REFRESH
4,content_5feee3994adb,client_7f2253d7e2,194,7812,181+,good,7812,stale_but_visible,REVIEW_REFRESH
5,content_c2d929d83eaa,client_7f2253d7e2,193,7558,181+,good,7558,stale_but_visible,REVIEW_REFRESH
6,content_b16bd7307b39,client_7f2253d7e2,194,4590,181+,good,4590,stale_but_visible,REVIEW_REFRESH
7,content_fe16a55cd13d,client_7f2253d7e2,194,4556,181+,good,4556,stale_but_visible,REVIEW_REFRESH
8,content_ecb6215e79fd,client_7f2253d7e2,194,4429,181+,good,4429,stale_but_visible,REVIEW_REFRESH
9,content_928af3e22c80,client_7f2253d7e2,193,1697,181+,moderate,1697,stale_but_visible,REVIEW_REFRESH



CSV written to: work/outputs/baseline_action_score.csv


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Top-20 review
# For every row: action, reason code, confidence note, and what could make it wrong.

top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    if row["score"] > 0:
        return "High rule confidence: both stale and visible conditions are met."
    return "High rule confidence: item fails the stale condition, so it is not selected."

def wrong_if(row):
    if row["score"] > 0:
        return (
            "Wrong if the page was recently updated despite the recorded "
            "staleness, or if impressions_90d does not reflect useful current visibility."
        )
    return (
        "Wrong only if the 180-day staleness threshold is too strict and "
        "recently updated high-visibility content should also be reviewed."
    )

top20["confidence_note"] = top20.apply(confidence_note, axis=1)
top20["what_would_make_it_wrong"] = top20.apply(wrong_if, axis=1)

review_columns = [
    "content_id",
    "action",
    "reason_code",
    "score",
    "confidence_note",
    "what_would_make_it_wrong"
]

top20_review = top20[review_columns].copy()

display(top20_review)

,content_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
0,content_cf56e2e2e282,REVIEW_REFRESH,stale_but_visible,61678,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
1,content_7368877ea310,REVIEW_REFRESH,stale_but_visible,59472,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
2,content_1bfaa38ff26c,REVIEW_REFRESH,stale_but_visible,25715,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
3,content_0a91db491d14,REVIEW_REFRESH,stale_but_visible,13299,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
4,content_5feee3994adb,REVIEW_REFRESH,stale_but_visible,7812,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
5,content_c2d929d83eaa,REVIEW_REFRESH,stale_but_visible,7558,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
6,content_b16bd7307b39,REVIEW_REFRESH,stale_but_visible,4590,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
7,content_fe16a55cd13d,REVIEW_REFRESH,stale_but_visible,4556,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
8,content_ecb6215e79fd,REVIEW_REFRESH,stale_but_visible,4429,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...
9,content_928af3e22c80,REVIEW_REFRESH,stale_but_visible,1697,High rule confidence: both stale and visible c...,Wrong if the page was recently updated despite...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

The weakest examples in the displayed top-20 review are the zero-score rows with high impressions but fewer than 180 days since the last update. They appear near the cutoff because they have strong visibility, but the rule correctly gives them no action because they do not meet the staleness condition.

The main limitation is that the 180-day threshold may be too strict and could miss recently updated pages that still deserve review.

Leakage check: the baseline score uses only days_since_last_update and impressions_90d. No future-window, label-derived, or outcome field is used.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak picks + leakage check

# Rows in the top 20 that have zero score are useful weak/non-selected examples.
weak_picks = top20_review[top20_review["score"] == 0].copy()

print("WEAK / NON-SELECTED PICKS")
display(weak_picks)

print("\nLEAKAGE CHECK")

# These are the only columns used to create the baseline score.
rule_inputs = [
    "days_since_last_update",
    "impressions_90d"
]

print("Rule inputs:", rule_inputs)

# Explicitly confirm that no future-window or label-derived field is used.
future_or_label_inputs = [
    col for col in rule_inputs
    if any(term in col.lower() for term in [
        "future",
        "label",
        "outcome",
        "next",
        "target"
    ])
]

print("Future/label-derived rule inputs:", future_or_label_inputs)

if len(future_or_label_inputs) == 0:
    print("LEAKAGE CHECK: PASS — no future-window or label-derived inputs are used.")
else:
    print("LEAKAGE CHECK: FAIL — inspect the rule inputs.")

WEAK / NON-SELECTED PICKS


,content_id,action,reason_code,score,confidence_note,what_would_make_it_wrong
17,content_5fe46e04994d,NO_ACTION,not_selected,0,High rule confidence: item fails the stale con...,Wrong only if the 180-day staleness threshold ...
18,content_aaef01a50def,NO_ACTION,not_selected,0,High rule confidence: item fails the stale con...,Wrong only if the 180-day staleness threshold ...
19,content_8c19996aa890,NO_ACTION,not_selected,0,High rule confidence: item fails the stale con...,Wrong only if the 180-day staleness threshold ...



LEAKAGE CHECK
Rule inputs: ['days_since_last_update', 'impressions_90d']
Future/label-derived rule inputs: []
LEAKAGE CHECK: PASS — no future-window or label-derived inputs are used.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

Self-check completed.

- All four sections are filled with both reasoning and supporting code.
- The notebook runs from top to bottom without errors.
- No client names, URLs, or private queries are included.
- The rule uses careful, decision-support language.
- The baseline uses only pre-decision signals.
- The ranked queue is generated by the notebook at work/outputs/baseline_action_score.csv.
- The notebook is ready to commit under work/notebooks/.